In [ ]:
import time
from collections import deque
from IPython.display import display, HTML, clear_output

class MaquinaDePostVisualDashboard:
    """
    Simulador Dinâmico da Máquina de Post para a linguagem a^n b^n c^n.
    Interface gráfica compatível com o simulador do Autômato de Pilha.
    """
    def __init__(self):
        self.transicoes = {}
        self.fila = deque()
        self.configurar_transicoes()

    def configurar_transicoes(self):
        self.transicoes[('q_val_a', 'a')] = ('q_val_a2', ['a'])
        self.transicoes[('q_val_a2', 'a')] = ('q_val_a2', ['a'])
        self.transicoes[('q_val_a2', 'b')] = ('q_val_b', ['b'])
        self.transicoes[('q_val_b', 'b')] = ('q_val_b', ['b'])
        self.transicoes[('q_val_b', 'c')] = ('q_val_c', ['c'])
        self.transicoes[('q_val_c', 'c')] = ('q_val_c', ['c'])
        self.transicoes[('q_val_c', '#')] = ('q_cancel_start', ['#'])

        self.transicoes[('q_cancel_start', 'a')] = ('q_skip_a', [])
        self.transicoes[('q_cancel_start', '#')] = ('q_aceita', [])
        self.transicoes[('q_skip_a', 'a')] = ('q_skip_a', ['a'])
        self.transicoes[('q_skip_a', 'b')] = ('q_skip_b', [])
        self.transicoes[('q_skip_b', 'b')] = ('q_skip_b', ['b'])
        self.transicoes[('q_skip_b', 'c')] = ('q_skip_c', [])
        self.transicoes[('q_skip_c', 'c')] = ('q_skip_c', ['c'])
        self.transicoes[('q_skip_c', '#')] = ('q_cancel_start', ['#'])

    def obter_simbolos_esperados(self, estado_atual):
        return sorted(list(set([s for (q, s) in self.transicoes.keys() if q == estado_atual])))

    def desenhar_estrutura(self, passo, estado, proximo_estado, simbolos_esperados, aceito=None):
        cor_fundo = "#f8f9fa"
        if aceito is True:
            cor_fundo = "#d4edda"
            simbolos_esperados = []
        elif aceito is False:
            cor_fundo = "#f8d7da"
            simbolos_esperados = []
            
        if simbolos_esperados:
            formatados = [f"<code style='background: #e9ecef; border: 1px solid #ced4da; padding: 4px 8px; border-radius: 4px; color: #198754; font-weight: bold; font-size: 1.1em;'>{s}</code>" for s in simbolos_esperados]
            esperados_html = " ".join(formatados)
        else:
            esperados_html = "<span style='color: #dc3545; font-style: italic; font-weight: bold; font-size: 0.9em;'>Nenhum (Parada)</span>"

        badge_canto_esquerdo = f"""
        <div style="position: absolute; top: 20px; left: 20px; background: #ffffff; border: 2px solid #adb5bd; border-radius: 8px; padding: 12px; box-shadow: 0 4px 8px rgba(0,0,0,0.1); width: 150px; text-align: center; z-index: 10;">
            <div style="font-size: 0.75em; color: #495057; font-weight: 800; margin-bottom: 8px; text-transform: uppercase; letter-spacing: 0.5px; border-bottom: 1px solid #dee2e6; padding-bottom: 6px;">
                Símbolos Válidos<br><span style="color: #d63384;">({estado})</span>
            </div>
            <div style="display: flex; gap: 6px; flex-wrap: wrap; justify-content: center;">
                {esperados_html}
            </div>
        </div>
        """

        fila_html = "<div style='display: flex; align-items: center; justify-content: center; gap: 6px; margin: 15px 0; padding: 5px; overflow-x: auto;'>"
        for idx, item in enumerate(self.fila):
            is_frente = (idx == 0)
            is_fim = (idx == len(self.fila) - 1 and len(self.fila) > 1)
            
            if is_frente and aceito is None:
                cor_item = "#ffc107"
                rotulo = "<span style='display:block; font-size:0.55em; color:#856404; font-weight:bold; margin-top:2px;'>LÊ DAQUI</span>"
            elif is_fim and aceito is None:
                cor_item = "#bde0fe" 
                rotulo = "<span style='display:block; font-size:0.55em; color:#003049; font-weight:bold; margin-top:2px;'>ESCREVE AQUI</span>"
            elif item == '#':
                cor_item = "#ffeeba" 
                rotulo = "<span style='display:block; font-size:0.55em; color:#7f6000; font-weight:normal; margin-top:2px;'>MARCADOR</span>"
            else:
                cor_item = "#e9ecef"
                rotulo = "<span style='display:block; font-size:0.55em; color:#6c757d; font-weight:normal; margin-top:2px;'>&nbsp;</span>"

            fila_html += f"""
                <div style="border: 2px solid #495057; padding: 6px 12px; text-align: center; font-size: 1.3em; background-color: {cor_item}; font-weight: bold; border-radius: 6px; min-width: 50px; box-shadow: 0 2px 4px rgba(0,0,0,0.05); line-height: 1.1;">
                    {item}
                    {rotulo}
                </div>
            """
            if idx < len(self.fila) - 1:
                fila_html += "<span style='color: #6c757d; font-weight: bold; font-size: 1.1em;'>-</span>"
        
        if not self.fila:
            fila_html += "<span style='color: #dc3545; font-style: italic; font-weight: bold;'>A Fila está vazia!</span>"
        fila_html += "</div>"
        
        if aceito is None:
            cor_destino = "#0dcaf0" if proximo_estado != 'q_rejeita' else "#dc3545"
            texto_destino = f"➔ {proximo_estado}"
        else:
            cor_destino = "#198754" if aceito else "#dc3545"
            texto_destino = "Parada Final"

        linha_destino = f"""
        <tr>
            <td style="padding: 4px 0; text-align: center;"><b>Transição Prevista (Destino):</b> <code style="font-size: 1.1em; background: #e9ecef; padding: 2px 8px; border-radius: 4px; color: {cor_destino}; font-weight: bold;">{texto_destino}</code></td>
        </tr>
        """
        
        html_final = f"""
        <div style="font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif; background-color: {cor_fundo}; padding: 25px; border-radius: 12px; width: 720px; position: relative; border: 1px solid #dee2e6; box-shadow: 0 6px 12px rgba(0,0,0,0.1); margin: 0 auto;">
            
            {badge_canto_esquerdo}

            <h3 style="margin-top: 10px; margin-bottom: 20px; color: #212529; text-align: center; border-bottom: 2px solid #dee2e6; padding-bottom: 10px;">Simulador Dinâmico: Máquina de Post</h3>
            
            <table style="width: 100%; border-collapse: collapse; margin-top: 10px; font-size: 1.05em;">
                <tr>
                    <td style="padding: 4px 0; text-align: center;"><b>Passo de Execução:</b> <span style="color: #0d6efd; font-weight:bold; font-size: 1.1em;">{passo:02d}</span></td>
                </tr>
                {linha_destino}
            </table>
            
            <h4 style="text-align: center; margin: 30px 0 5px 0; color: #495057; font-weight: 600; border-top: 1px dashed #ced4da; padding-top: 15px;">ESTRUTURA DA FILHA (FIFO)</h4>
            {fila_html}
            <div style="text-align: center; margin-top: 5px; color: #6c757d; font-size: 0.85em;">A máquina lê do início (amarelo) e reescreve no fim (azul)</div>
        </div>
        """
        
        clear_output(wait=True)
        display(HTML(html_final))
        time.sleep(1.8) 

    def processar(self, string_entrada):
        self.fila = deque(list(string_entrada) + ['#'])
        estado_atual = 'q_val_a'
        passo = 0

        while estado_atual not in ['q_aceita', 'q_rejeita']:
            simbolos_esperados = self.obter_simbolos_esperados(estado_atual)

            if not self.fila:
                self.desenhar_estrutura(passo, estado_atual, 'q_rejeita', simbolos_esperados, aceito=False)
                break

            simbolo_frente = self.fila[0]
            transicao_futura = self.transicoes.get((estado_atual, simbolo_frente))
            proximo_estado_previsto = transicao_futura[0] if transicao_futura else 'q_rejeita'

            self.desenhar_estrutura(passo, estado_atual, proximo_estado_previsto, simbolos_esperados)

            simbolo_lido = self.fila.popleft()
            chave = (estado_atual, simbolo_lido)

            if chave in self.transicoes:
                proximo_estado, simbolos_escrever = self.transicoes[chave]
                for s in simbolos_escrever:
                    self.fila.append(s)
                estado_atual = proximo_estado
            else:
                estado_atual = 'q_rejeita'
                self.fila.appendleft(simbolo_lido) 
                break
            
            passo += 1

        if estado_atual == 'q_aceita':
            self.desenhar_estrutura(passo, estado_atual, None, [], aceito=True)
        else:
            self.desenhar_estrutura(passo, estado_atual, None, [], aceito=False)

In [ ]:
mp_visual_1 = MaquinaDePostVisualDashboard()

mp_visual_1.processar("aabbcc")

Passo de Execução: 19
Transição Prevista (Destino): Parada Final


In [3]:
mp_visual_2 = MaquinaDePostVisualDashboard()

mp_visual_2.processar("aabbc")

Passo de Execução: 14
Transição Prevista (Destino): Parada Final


In [4]:
mp_visual_3 = MaquinaDePostVisualDashboard()

mp_visual_3.processar("abca")

Passo de Execução: 03
Transição Prevista (Destino): Parada Final
